# Continuous optimization (CALM)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abel-ai-lab/andrey/blob/main/examples/calm.ipynb)

CALM learns a weighted DAG by gradient descent instead of searching over graphs. It first estimates
the moral graph, which leaves only the pairs of variables that can share an edge. It then
minimizes the Gaussian likelihood plus a penalty per edge, under a constraint that the graph is
acyclic. The fit runs on torch, so it needs the `torch` extra.

## Start

Choose a CPU runtime and run this cell first. In Colab it installs `andrey-core==0.6.0` from PyPI;
local kernels use the installed environment. CALM needs torch, which Colab provides; the next cell
installs it with uv in a local kernel that lacks it.

In [1]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.environ["ANDREY_DEVICE"] = "cpu"
    !pip install --quiet "andrey-core[viz]==0.6.0" pandas

In [2]:
from importlib.util import find_spec

if not IN_COLAB and find_spec("torch") is None:
    !uv pip install --python {sys.executable} "torch>=2.2"

In [3]:
import numpy as np
from IPython.display import display

import andrey
import andrey.metrics as metrics
import andrey.viz
from andrey.data import benchmarks

## Data

`linear_gauss_er` draws an Erdos-Renyi graph with linear mechanisms and Gaussian noise: here 10
variables and 500 samples.

In [4]:
seed = 1
dataset = benchmarks.scm("linear_gauss_er", d=10).sample(n=500, seed=seed, scale="standardize")
X, truth = dataset.data, dataset.observed_graph
print(f"{X.shape[0]} observations, {X.shape[1]} variables, seed {seed}")
display(andrey.viz.draw(truth))

500 observations, 10 variables, seed 1


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 464 316" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="464" height="316" fill="#fafbfc"/><path d="M 211.35 139.10 Q 116.08 185.83 40.47 273.94" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="220.33,134.69 209.15,134.61 213.55,143.59" fill="#3a4658" stroke-linejoin="round"/><line x1="209.06" y1="127.30" x2="44.97" y2="115.39" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.03,128.03 209.42,122.32 208.70,132.29" fill="#3a4658" stroke-linejoin="round"/><line x1="211.30" y1="118.95" x2="143.70" y2="86.24" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="220.30,123.31 213.47,114.45 209.12,123.45" fill="#3a4658" stroke-linejoin="round"/><line x1="245.00" y1="128.97" x2="309.00" y2="128.97" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.00,128.97 309.00,123.97 309.00,133.97" fill="#3a4658" stroke-linejoin="round"/><line x1="244.34" y1="124.89" x2="410.16" y2="70.06" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.66,66.92 408.59,65.31 411.73,74.81" fill="#3a4658" stroke-linejoin="round"/><line x1="42.60" y1="276.28" x2="113.24" y2="226.15" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="121.40,220.36 110.35,222.07 116.14,230.23" fill="#3a4658" stroke-linejoin="round"/><line x1="45.00" y1="283.81" x2="109.00" y2="283.81" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.00,283.81 109.00,278.81 109.00,288.81" fill="#3a4658" stroke-linejoin="round"/><line x1="41.27" y1="123.57" x2="115.60" y2="196.71" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="122.73,203.72 119.11,193.14 112.10,200.27" fill="#3a4658" stroke-linejoin="round"/><path d="M 42.53 106.83 Q 122.11 49.28 209.27 35.73" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.15,34.19 208.50,30.79 210.04,40.67" fill="#3a4658" stroke-linejoin="round"/><line x1="38.61" y1="125.65" x2="120.31" y2="264.00" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="125.39,272.61 124.61,261.46 116.00,266.54" fill="#3a4658" stroke-linejoin="round"/><line x1="44.31" y1="110.28" x2="110.22" y2="87.96" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.69,84.75 108.61,83.22 111.82,92.70" fill="#3a4658" stroke-linejoin="round"/><line x1="44.89" y1="112.79" x2="409.19" y2="65.78" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.11,64.50 408.55,60.82 409.83,70.74" fill="#3a4658" stroke-linejoin="round"/><line x1="209.00" y1="212.84" x2="145.00" y2="212.84" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,212.84 209.00,207.84 209.00,217.84" fill="#3a4658" stroke-linejoin="round"/><line x1="211.30" y1="42.21" x2="143.70" y2="74.92" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="220.30,37.86 209.12,37.71 213.47,46.71" fill="#3a4658" stroke-linejoin="round"/><line x1="342.84" y1="121.80" x2="412.82" y2="75.53" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="421.16,70.01 410.06,71.36 415.57,79.70" fill="#3a4658" stroke-linejoin="round"/><circle cx="232.00" cy="128.97" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="128.97" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">0</text><circle cx="32.00" cy="283.81" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="283.81" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">1</text><circle cx="32.00" cy="114.45" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="114.45" text-anchor="middle" font-size="11" font-weight="600" fill="

## Fit CALM

`andrey.calm` returns a weighted DAG. Its optimizer draws random numbers and is seeded on every
call, so a run repeats on one machine. The fit takes 10 to 30 seconds on a CPU.

In [5]:
out = andrey.calm(X)
print(f"{out.structure.kind}, {out.metadata['n_edges']} edges, seed {out.metadata['seed']}")
display(andrey.viz.draw(out.structure))

dag, 12 edges, seed 0


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 764 240" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="764" height="240" fill="#fafbfc"/><line x1="245.00" y1="85.53" x2="309.00" y2="86.56" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.00,86.73 309.08,81.57 308.92,91.56" fill="#3a4658" stroke-linejoin="round"/><line x1="210.11" y1="78.26" x2="144.37" y2="57.06" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.63,81.33 211.65,73.50 208.58,83.02" fill="#3a4658" stroke-linejoin="round"/><line x1="209.00" y1="85.51" x2="45.00" y2="86.83" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,85.43 208.96,80.51 209.04,90.51" fill="#3a4658" stroke-linejoin="round"/><line x1="244.96" y1="157.28" x2="609.07" y2="185.18" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="619.04,185.94 609.45,180.19 608.69,190.16" fill="#3a4658" stroke-linejoin="round"/><line x1="244.88" y1="154.52" x2="609.21" y2="104.58" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="619.12,103.22 608.53,99.62 609.89,109.53" fill="#3a4658" stroke-linejoin="round"/><line x1="242.68" y1="148.88" x2="313.10" y2="100.04" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="321.32,94.34 310.25,95.93 315.95,104.15" fill="#3a4658" stroke-linejoin="round"/><line x1="541.19" y1="96.13" x2="615.74" y2="170.67" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="622.81,177.74 619.27,167.14 612.20,174.21" fill="#3a4658" stroke-linejoin="round"/><line x1="509.00" y1="86.94" x2="445.00" y2="86.94" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="519.00,86.94 509.00,81.94 509.00,91.94" fill="#3a4658" stroke-linejoin="round"/><line x1="544.87" y1="88.80" x2="609.24" y2="98.15" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="619.13,99.58 609.96,93.20 608.52,103.10" fill="#3a4658" stroke-linejoin="round"/><line x1="709.00" y1="186.94" x2="645.00" y2="186.94" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="719.00,186.94 709.00,181.94 709.00,191.94" fill="#3a4658" stroke-linejoin="round"/><line x1="409.00" y1="86.94" x2="345.00" y2="86.94" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.00,86.94 409.00,81.94 409.00,91.94" fill="#3a4658" stroke-linejoin="round"/><line x1="110.22" y1="60.44" x2="44.31" y2="82.76" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.69,57.24 108.61,55.71 111.82,65.18" fill="#3a4658" stroke-linejoin="round"/><circle cx="232.00" cy="85.32" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="85.32" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">0</text><circle cx="232.00" cy="156.29" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="156.29" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">1</text><circle cx="532.00" cy="86.94" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="532.00" y="86.94" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">2</text><circle cx="732.00" cy="186.94" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="732.00" y="186.94" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">3</text><circle cx="632.00" cy="186.94" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="632.00" y="186.94" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">4</text><circle cx="432.00" cy="86.94" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="432.00" y="86.94" text-anchor="middle" font-size="11" font-weight="600" f

## Check accuracy

SHD counts errors against the true graph; lower is better. Gaussian data identify the graph only up
to its equivalence class, so `mec_shd` compares the two CPDAGs. The empty graph's SHD is the number
of true edges, so an estimate at or above it has not improved on returning no edges.

In [6]:
d = truth.n_nodes
empty = andrey.GraphStructure.from_numpy(np.zeros((d, d), dtype=np.uint8), kind="dag")
print("CALM SHD:", metrics.score(out, truth)["mec_shd"])
print("Empty graph SHD:", metrics.score(empty, truth)["mec_shd"])

CALM SHD: 11
Empty graph SHD: 15


## Cost and limits

Every optimizer step computes a determinant and a matrix power of the full weight matrix, so a
step's cost grows with the cube of the variable count. The median fit in the `andrey.calm`
docstring took 18 s at 10 variables and 59 s at 50, on 4 cores. The fit is nonconvex and can stop
at a local optimum, and one dataset can favor any method: change `seed`, `d`, or `n` and rerun.